# From FERPA Data to Reports, Colab Edition

A no-code-friendly, end-to-end notebook that transfers a real student-evaluation pipeline to **any** FERPA-protected dataset with structured fields + free-text.

**Six stages:** ingest → standardize → reconcile identity → AI insight (cached) → render report → review & distribute.

**Before you run anything:**
1. Open the **🔑 Secrets** panel in the left sidebar.
2. Add three secrets and toggle *Notebook access* ON for each:
   - `LLM_SANDBOX_BASE_URL`, your institution's gateway URL, ending in `/v1`
   - `LLM_SANDBOX_API_KEY`, the key your gateway admin gave you
   - `LLM_SANDBOX_MODEL`, e.g. `claude-v4.5-haiku`
3. **Never paste your key into a cell.** Secrets live against your Google account, not in this file.

> ⚠️ **FERPA:** Confirm your institution permits this data class in Colab, and remember, only the *free-text* is ever sent to the gateway. Identifiers stay in the structured layer. Do a dry run on synthetic data first.

## Cell 1, Setup: install + read secrets

In [ ]:
!pip install openai pandas thefuzz reportlab --quiet

from google.colab import userdata
from openai import OpenAI

# The gateway authenticates with the x-api-key header (not the bearer token the
# OpenAI SDK sends by default), so we pass the key in BOTH places.
client = OpenAI(
    base_url=userdata.get("LLM_SANDBOX_BASE_URL"),
    api_key=userdata.get("LLM_SANDBOX_API_KEY"),
    default_headers={"x-api-key": userdata.get("LLM_SANDBOX_API_KEY")},
)
MODEL = userdata.get("LLM_SANDBOX_MODEL") or "claude-v4.5-haiku"
print("Gateway configured. Model:", MODEL)

## Cell 1b, Smoke test (optional, ~1.3K tokens)
Confirms your secrets and gateway work before you process real data.

In [ ]:
test = client.chat.completions.create(
    model=MODEL, max_tokens=128,
    messages=[
        {"role": "system", "content": "You are a helpful assistant specialized in analyzing feedback."},
        {"role": "user",   "content": "[1] Great instructor.\n[2] Office hours were hard to reach."},
        {"role": "user",   "content": "Give a one-line keyword summary with [n] citations. No markdown."},
    ],
)
print(test.choices[0].message.content)
print("tokens used:", test.usage.total_tokens)

## Cell 2, Ingest & standardize (stages 1 to 2)

Upload your messy export. We build a **clean layer on top** of it and never edit the source.
Edit the `rename(...)` map to match whatever your file actually calls its columns.

*No file handy? Run the next cell to generate a synthetic advising-notes CSV for a dry run.*

In [ ]:
# OPTIONAL: generate synthetic data for a safe dry run (no real records).
import pandas as pd
synthetic = pd.DataFrame({
    "Advisor (full name)": ["Garcia, Maria"]*3 + ["Okoro, Daniel"]*3 + ["Chen, Wei"]*2,
    "Term": ["F25"]*8,
    "Notes / summary": [
        "Student felt well supported choosing a major; clear next steps given.",
        "Helped map out a graduation plan; student left reassured.",
        "Discussed internship options; advisor very responsive over email.",
        "Great listener, but follow-up email took two weeks to arrive.",
        "Patient walkthrough of degree audit; student understood requirements.",
        "Encouraging session; connected student to tutoring resources.",
        "Helpful on course selection; wished the meeting were longer.",
        "Clarified financial aid timeline; student grateful for the detail.",
    ],
    # Structured KPI columns (numbers) -> these belong in CHARTS, not the AI summary.
    "Satisfaction (1-5)":   [5, 5, 4, 3, 5, 5, 4, 5],
    "Response time (hrs)":  [2, 1, 3, 72, 24, 12, 5, 4],
    "Session length (min)": [35, 40, 30, 30, 40, 35, 25, 30],
})
synthetic.to_csv("advising_notes_sample.csv", index=False)
print("Wrote advising_notes_sample.csv, upload it in the next cell, or use your own.")

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()                       # pick your CSV (or the sample above)
raw = pd.read_csv(next(iter(uploaded)))

# Standardize: map the export's column names to clean ones. EDIT THESE to match your file.
df = raw.rename(columns={
    "Advisor (full name)": "subject",   # the thing each report is about
    "Term":                "period",    # the reporting period (quarter/term/cycle)
    "Notes / summary":     "text",      # the free-text the AI summarizes
})[["subject", "period", "text"]].dropna(subset=["text"])
df.head()

## Cell 3, Reconcile identity (stage 3)

Fuzzy-match each subject to a **roster of truth** so typos and aliases collapse to one canonical name. Unmatched rows are dropped (and printed, so you can fix the roster).

In [ ]:
from thefuzz import process

# Your canonical list of subjects. EDIT THIS.
ROSTER = ["Garcia, Maria", "Okoro, Daniel", "Chen, Wei"]
THRESHOLD = 85   # 85 balances precision vs. recall; lower lets in false matches

def canonical(name):
    match, score = process.extractOne(str(name), ROSTER)
    return match if score >= THRESHOLD else None

df["subject_matched"] = df["subject"].map(canonical)
dropped = df[df["subject_matched"].isna()]["subject"].unique()
if len(dropped):
    print("Unmatched (add to ROSTER if real):", list(dropped))
df = df.dropna(subset=["subject_matched"]).copy()
df["subject"] = df["subject_matched"]
df[["subject", "period", "text"]].head()

## Cell 4, AI insight, cached (stage 4)

Group by (subject, period), and for each group either read the cached insight (**0 tokens**) or call the gateway once and cache the result. Only free-text, numbered `[1]`, `[2]` so the insight can cite it, ever leaves the notebook.

In [ ]:
import sqlite3, datetime

db = sqlite3.connect("ai_cache.sqlite")
db.execute("""CREATE TABLE IF NOT EXISTS ai_cache(
    subject TEXT, period TEXT, analysis TEXT, total_tokens INTEGER, created_at TEXT,
    PRIMARY KEY (subject, period))""")

SYSTEM_PROMPT = "You are a helpful assistant specialized in analyzing feedback."
TASK_PROMPT = (
    "Using the provided comments, generate a concise keyword summary (each keyword "
    "one to two words, color-coded Green/Orange/Red strictly by the actual tone; lean "
    "positive) and a one-to-two paragraph insight no longer than the comments. Add "
    "nothing not present and never invent problems. Comments are numbered [1], [2]...; "
    "in the insight, cite the comments supporting each statement inline, e.g. [2] or "
    "[3][7]. Only cite numbers that appear. Do not use markdown. "
    "Format: 'Keyword Summary: kw1 (color), kw2 (color); Feedback Insight: para1 para2'"
)
MIN_FOR_AI = 5   # too few comments => unreliable sentiment; skip

def analyze(texts):
    numbered = "\n".join(f"[{i}] {t}" for i, t in enumerate(texts, 1))
    r = client.chat.completions.create(
        model=MODEL, max_tokens=1024,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": numbered},
            {"role": "user",   "content": TASK_PROMPT},
        ],
    )
    return r.choices[0].message.content, r.usage.total_tokens

def get_or_analyze(subject, period, texts):
    hit = db.execute("SELECT analysis FROM ai_cache WHERE subject=? AND period=?",
                     (subject, period)).fetchone()
    if hit:                                     # cache HIT -> 0 tokens
        return hit[0]
    analysis, tokens = analyze(texts)           # cache MISS -> one gateway call
    db.execute("INSERT OR REPLACE INTO ai_cache VALUES (?,?,?,?,?)",
               (subject, period, analysis, tokens,
                datetime.datetime.now().isoformat(timespec="seconds")))
    db.commit()
    return analysis

insights = {}
for (subject, period), grp in df.groupby(["subject", "period"]):
    if len(grp) >= MIN_FOR_AI:
        insights[(subject, period)] = get_or_analyze(subject, period, list(grp["text"]))
    else:
        print(f"Skipped {subject} ({period}): only {len(grp)} comment(s).")

spent = db.execute("SELECT COALESCE(SUM(total_tokens),0) FROM ai_cache").fetchone()[0]
print(f"\n{len(insights)} insight(s) ready. Total tokens ever spent (this cache): {spent}")
for k, v in insights.items():
    print("\n===", k, "===\n", v)

## Cell 5, Render the report (stages 5 to 6)

One PDF per subject/period. A human reviews these before anyone else sees them, the AI never has release authority.

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from google.colab import files
import re

styles = getSampleStyleSheet()
for (subject, period), text in insights.items():
    safe = re.sub(r"[^A-Za-z0-9]+", "_", f"{subject}_{period}").strip("_")
    fname = f"report_{safe}.pdf"
    SimpleDocTemplate(fname).build([
        Paragraph(f"Summary Report, {subject} ({period})", styles["Title"]),
        Spacer(1, 12),
        Paragraph(text.replace("\n", "<br/>"), styles["BodyText"]),
        Spacer(1, 18),
        Paragraph("AI insight powered by LLM Sandbox. Draft for human review, "
                  "not for distribution until approved.", styles["Italic"]),
    ])
    print("Wrote", fname)
    files.download(fname)

## Cell 6, Bonus: chart a KPI (consistent, code-generated)

Free text became the AI summary. Your **numbers** (the KPI columns) belong in a chart. The chart is *code*, so it is consistent: same data, same chart, every time. To change the KPI or the color, ask Gemini, never ask AI to "draw" a chart.

In [ ]:
import matplotlib.pyplot as plt

KPI       = "Satisfaction (1-5)"      # any numeric column in your file
SUBJECT   = "Advisor (full name)"     # your subject column (the original name in the upload)
BAR_COLOR = "#003660"                 # navy; pick any HTML hex color (e.g. #1178b5, #febc11)

agg = raw.groupby(SUBJECT)[KPI].mean().sort_values(ascending=False)
ax = agg.plot(kind="bar", color=BAR_COLOR, figsize=(7, 4))
ax.set_title(f"Average {KPI} by {SUBJECT}")
ax.set_ylabel(KPI); ax.set_xlabel("")
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

---
## That's the whole machine

Five cells. To point it at a different FERPA dataset, change only:
- the **column map** in Cell 2 (`subject` / `period` / `text`),
- the **roster** in Cell 3.

The gateway call in Cell 4 never changes. Re-running is free for anything already in `ai_cache.sqlite`.

**Three principles to carry:** build on top (never mutate the source) · cache hard-won output · let the system read its own runs.

*Derived from a real student-evaluation pipeline. Built on UCSB's LLM Sandbox, with thanks to Yaheya & the LLM Sandbox team. This is an architecture guide, not legal advice; verify data-governance terms with your institution before sending real records.*